In [1]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_295K_278464_17O_opt_magres.magres')

In [2]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [3]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [4]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [5]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-51.76216646 165.84862613 -79.15415126]
 [172.92109001  54.32180818  28.48008084]
 [  8.98456411 -24.91510788 -46.19466476]]

17O2 sigma:
 [[ -51.76216646 -165.84862613   79.15415126]
 [-172.92109001   54.32180818   28.48008084]
 [  -8.98456411  -24.91510788  -46.19466476]]

17O3 sigma:
 [[-51.76216646 165.84862613  79.15415126]
 [172.92109001  54.32180818 -28.48008084]
 [ -8.98456411  24.91510788 -46.19466476]]

17O4 sigma:
 [[ -51.76216646 -165.84862613  -79.15415126]
 [-172.92109001   54.32180818  -28.48008084]
 [   8.98456411   24.91510788  -46.19466476]]

17O5 sigma:
 [[ -40.78847648  225.05282945   53.92102317]
 [ 194.61431093  106.55726755  -87.20255622]
 [  16.8554785   -49.92206346 -171.92816106]]

17O6 sigma:
 [[ -40.78847648 -225.05282945  -53.92102317]
 [-194.61431093  106.55726755  -87.20255622]
 [ -16.8554785   -49.92206346 -171.92816106]]

17O7 sigma:
 [[ -40.78847648  225.05282945  -53.92102317]
 [ 194.61431093  106.55726755   87.20255622]
 [ -16.8554785 

In [6]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.2791786921842325

17O2 sigma:
 6.279178692184241

17O3 sigma:
 6.279178692184209

17O4 sigma:
 6.2791786921842005

17O5 sigma:
 8.255981226978793

17O6 sigma:
 8.255981226978772

17O7 sigma:
 8.255981226978804

17O8 sigma:
 8.25598122697881



In [7]:
Q = -0.0256 #electric quadrupole moment for O17 in barn

CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + l = 1 + l = 2) Tensor from magres
CS_total[0,0] = -51.7622; CS_total[0,1] = 165.8486; CS_total[0,2] = -79.1542;
CS_total[1,0] = 172.9211; CS_total[1,1] =  54.3218; CS_total[1,2] = 28.4801;
CS_total[2,0] = 8.9846; CS_total[2,1] = -24.9151; CS_total[2,2] =  -46.1947;

Cs = np.zeros((3,3)) # CS symmetric (l = 0 + l = 2) 

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)
efg[0,0]= -0.1464; efg[0,1]= 0.1389; efg[0,2]=  -0.7403;
efg[1,0]= efg[0,1]; efg[1,1]= 0.0753; efg[1,2]= 0.6308;
efg[2,0]= efg[0,2]; efg[2,1]= efg[1,2]; efg[2,2]= 0.0711;

# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 

V = efg*Q*234.9647

print(V)

print(Cs)

[[ 0.8806101  -0.83549688  4.45297581]
 [-0.83549688 -0.45293675 -3.79432276]
 [ 4.45297581 -3.79432276 -0.42767335]]
[[-51.7622  169.38485 -35.0848 ]
 [169.38485  54.3218    1.7825 ]
 [-35.0848    1.7825  -46.1947 ]]


In [8]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 6.28371808 -0.71454277 -5.56917531] 

 Unsorted Eigenvectors:
 [[-0.60950455  0.6492712  -0.4549188 ]
 [ 0.44509855  0.75510471  0.48135657]
 [-0.65604229 -0.09090532  0.74922943]] 

Sorted Eigenvalues: 
 [-0.71454277 -5.56917531  6.28371808] 

Sorted Eigenvectors: 
 [[ 0.6492712  -0.4549188  -0.60950455]
 [ 0.75510471  0.48135657  0.44509855]
 [-0.09090532  0.74922943 -0.65604229]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 180.44302572 -182.57146917  -41.50665655] 

 Unsorted Eigenvectors:
 [[ 0.59558919  0.79424387 -0.12020895]
 [ 0.79868107 -0.56949852  0.19437076]
 [-0.08591896  0.21177374  0.9735347 ]] 

Sorted Eigenvalues: 
 [ -41.50665655 -182.57146917  180.44302572] 

Sorted Eigenvectors: 
 [[-0.12020895  0.79424387  0.59558919]
 [ 0.19437076 -0.56949852  0.79868107]
 [ 0.9735347   0.21177374 -0.08591896]] 



In [9]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.7145427709067971 -5.569175307267247 6.2837180781740445
CSA Tensor Components δyy, δxx, δzz: 
 -41.506656548675956 -182.57146917350158 180.44302572217745


In [10]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity           Value
------------  ----------
CQ (MHz)        6.28372
etaq            0.772573
iso_cs (ppm)  -14.545
csa (ppm)     194.988
etas            0.723454


In [11]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.4549188   0.6492712  -0.60950455]
 [ 0.48135657  0.75510471  0.44509855]
 [ 0.74922943 -0.09090532 -0.65604229]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
-6.917982178836094 130.99872901336533 36.139269169068264 

Direction cosine csa: 

[[ 0.79424387 -0.12020895  0.59558919]
 [-0.56949852  0.19437076  0.79868107]
 [ 0.21177374  0.9735347  -0.08591896]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
77.72759706263568 94.92887096999544 -53.287519701540106 



In [12]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: 28.650803592963182 chi: 87.20033430911018 xi: -87.18040482036584 



**Rotation of tensors Crystal--> Tenon Frame**

In [13]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[-51.7622  169.38485 -35.0848 ]
 [169.38485  54.3218    1.7825 ]
 [-35.0848    1.7825  -46.1947 ]]
CSA Tensor in Tenon Frame: 
 [[  92.79860305  -99.70003538 -118.71142854]
 [ -99.70003538  -52.37045305  -23.7587216 ]
 [-118.71142854  -23.7587216   -84.06325   ]]
Quad Tensor in Crystal Frame: 
 [[ 0.8806101  -0.83549688  4.45297581]
 [-0.83549688 -0.45293675 -3.79432276]
 [ 4.45297581 -3.79432276 -0.42767335]]
Quad Tensor in Tenon Frame: 
 [[ 0.14885271 -1.3207118   3.11045032]
 [-1.3207118  -4.82829689  1.21268923]
 [ 3.11045032  1.21268923  4.67944418]]
